In [1]:
%run utils.py

In [2]:
spark = get_spark_session(app_name="Streaming")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0633572f-1791-4773-add1-c0556e7c22ee;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 218ms :: artifacts dl 6ms
	:: modules in use

SparkSession started with app name: Streaming


In [3]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json

cdc_schema = StructType([
    StructField("payload", StructType([
        StructField("after", StructType([
            StructField("id", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("status", StringType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])

# Read the stream from Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "192.168.1.74:9092") \
    .option("subscribe", "dbserver1.public.orders") \
    .option("startingOffsets", "earliest") \
    .load()

df_parsed = df_kafka.select(
    from_json(col("value").cast("string"), cdc_schema).alias("data")
).select(
    col("data.payload.after.id").cast("string").alias("id"),
    col("data.payload.after.product_name").alias("product_name"),
    col("data.payload.after.status").alias("status"),
    col("data.payload.op").alias("op"),
    col("data.payload.ts_ms").alias("ts_ms")
).filter(col("id").isNotNull()) # Hudi yêu cầu khóa chính không được null


table_name_cow = "orders"
base_path = f"s3a://raw/cdc_postgres/public"
checkpointLocation = f"s3a://raw/cdc_postgres/checkpoints"

cow_hudi_conf = {
    "hoodie.table.name": table_name_cow, # The name of our Hudi table.
    "hoodie.datasource.write.recordkey.field": "id", # The column that acts as the unique identifier for each record.
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE", # Hudi uses Copy-on-Write as the default table type, but we are being explicit here.
    "hoodie.datasource.write.partitionpath.field": "status", # The column Hudi uses to partition the data on storage.
    "hoodie.datasource.write.precombine.field": "ts_ms", # The field used to deduplicate records when a conflict occurs.
    "hoodie.table.cdc.enabled": "true",
    "hoodie.datasource.write.hive_style_partitioning": "true", # This ensures partition directories are named like `city=new_york`.
    "hoodie.datasource.hive_sync.mode": "hms",
    "hoodie.datasource.hive_sync.jdbcurl": "thrift://hive-metastore:9083",
    "hoodie.datasource.hive_sync.enable": "true",
    "hoodie.datasource.hive_sync.support_timestamp": "true",
    'hoodie.upsert.shuffle.parallelism': 2,
    'hoodie.insert.shuffle.parallelism': 2
}

# write stream to new hudi table
stream_query = df_parsed.writeStream.format("hudi") \
    .options(**cow_hudi_conf) \
    .outputMode("append") \
    .option("path", f"{base_path}/{table_name_cow}") \
    .option("checkpointLocation", checkpointLocation) \
    .trigger(processingTime='2 seconds') \
    .start()


# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. sun.jvm.hotspot.memory.Universe.getNarrowOopBase()


In [4]:
# spark.stop()